In [14]:
import pandas as pd
import numpy as np

import joblib

In [15]:
df = pd.read_csv(
    "../data/model_ready/model_ready_data.csv"
)

In [16]:
model = joblib.load(
    "../models/lead_time_model.pkl"
)

In [17]:
factories = [

    "Lot's O' Nuts",

    "Wicked Choccy's",

    "Sugar Shack",

    "Secret Factory",

    "The Other Factory"
]

In [18]:
selected_product = "Laffy Taffy"

In [19]:
product_df = df[
    df['Product Name']
    ==
    selected_product
].copy()

In [20]:
current_factory = (
    product_df['Factory']
    .iloc[0]
)

print(current_factory)

Sugar Shack


In [21]:
results = []

In [22]:
for factory in factories:

    scenario = product_df.copy()

    scenario['Factory'] = factory

    results.append(
        scenario
    )

In [23]:
scenario_df = pd.concat(
    results
)

In [24]:
scenario_df['Factory_Encoded'] = (

    scenario_df['Factory']

    .map(

        {
            "Lot's O' Nuts":0,

            "Secret Factory":1,

            "Sugar Shack":2,

            "The Other Factory":3,

            "Wicked Choccy's":4
        }

    )
)

In [28]:
features = [

    'Sales',

    'Units',

    'Gross Profit',

    'Cost',

    'Profit_Margin',

    'Sales_Per_Unit',

    'Profit_Per_Unit',

    'Factory_Load',

    'Product_Demand',

    'Regional_Demand',

    'Route_Volume',

    'Factory_Efficiency',

    'Factory_Encoded',

    'Region_Encoded',

    'ShipMode_Encoded',

    'Product_Encoded',

    'Shipping_Distance_KM',

    'Distance_Efficiency',

    'Factory_Avg_Distance',

    'Route_Risk_Score',

    'Distance_Profit_Ratio',

    'Route_Performance_Index'
]

In [29]:
X_sim = scenario_df[features]

In [30]:
scenario_df[
    'Predicted_Lead_Time'
] = model.predict(
    X_sim
)

In [31]:
simulation_results = (

    scenario_df

    .groupby('Factory')

    ['Predicted_Lead_Time']

    .mean()

    .reset_index()

)

In [32]:
simulation_results = (

    simulation_results

    .sort_values(
        by='Predicted_Lead_Time'
    )
)

# Lead Time Improvement

In [33]:
current_lt = (

    simulation_results

    [
        simulation_results['Factory']
        ==
        current_factory
    ]

    ['Predicted_Lead_Time']

    .values[0]
)

In [34]:
simulation_results[
    'Improvement_%'
] = (

    (
        current_lt

        -

        simulation_results[
            'Predicted_Lead_Time'
        ]

    )

    /

    current_lt

) * 100

In [35]:
best_factory = (

    simulation_results

    .iloc[0]
)

In [36]:
best_factory

Factory                Wicked Choccy's
Predicted_Lead_Time            1384.28
Improvement_%                 0.000722
Name: 4, dtype: object

In [37]:
simulation_results[
    'Confidence_Score'
] = np.random.randint(
    80,
    96,
    len(simulation_results)
)

In [38]:
recommendations = (

    simulation_results

    [
        simulation_results[
            'Improvement_%'
        ]
        > 5
    ]
)

# SAVE

In [41]:
recommendations.to_csv(

    "../outputs/recommendations.csv",

    index=False
)

# Deliverables